## COMP3366 Tutorial 1: Introduction to Qiskit

#### Tutor: Entong He (Inherited from our extraordinary former TA, Manwen Liao, with modifications due to qiskit language update)

This tutorial will introduce you to the basic usage of Qiskit. You can learn how to create a quantum circuit, run simulation and visualization.

---

### 0. Getting everything ready
Let's first prepare the environment we need. We need Qiskit for quantum programming and matplotlib for visualization. The following command will automatically install Qiskit and its dependencies, including NumPy and SciPy - some scientific computing packages you might have already been using for your machine learnign courses.

In [ ]:
%pip install qiskit qiskit-aer matplotlib numpy

Including the packages we need:

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector
from qiskit.quantum_info.operators import Operator
from qiskit.circuit.library import MCXGate
from qiskit.circuit.library import UnitaryGate
from qiskit.visualization import plot_bloch_vector
from qiskit.visualization import plot_bloch_multivector

### 1. Creating your first quantum circuit
Here we implement the first quantum circuit using Qiskit. The grammar is `QuantumCircuit(# of qubits, name tag)`

A single qubit quantum circuit is automatically initialized as $|0\rangle$, i.e., the ground state.

In [ ]:
circuit = QuantumCircuit(1)

Alternatively, you might want to manage your registers by grouping them when the circuit becomes complicated. You can do so by first creating registers, naming them, and instantiating the quantum circuit with the registers.

In [ ]:
registers = QuantumRegister(3, 'my example register')
circuit = QuantumCircuit(registers)

Now we measure the circuit output in the computational basis. The method `measure_all()` can measure all qubits under the computational basis.

In [ ]:
circuit.measure_all()

Now, the `circuit.draw()` method can visualize the circuit we just created.

In [ ]:
circuit.draw()

To run our circuit, we need a simulator. Aer is a high performance simulator for quantum circuits. It provides sevaral backends for different simulation goals. See the document [here](https://qiskit.org/documentation/apidoc/aer_provider.html) for details.

Here, we just use 'aer_simulator' for simple usage. For convenience, wrap them into a function.

In [ ]:
def simulate(circuit, shots=30000):
    backend = Aer.get_backend('aer_simulator')
    result = backend.run(circuit, shots=shots).result() 
    counts = result.get_counts()
    keys = counts.keys()
    for key in keys:
        counts[key] = counts[key] / shots
    return counts

In [ ]:
circuit = QuantumCircuit(1)
circuit.h(0)
circuit.measure_all()
hist_outcome = simulate(circuit)
plot_histogram(hist_outcome)

You can, of course, initialize the input qubit in a given state, as long as you provide the state vector to the `initialize` method.

Prototype: `.initialize(initial_state, index of qubit)`.

In [ ]:
initial_state = [1/np.sqrt(2), 1/np.sqrt(2)]  # Define the state vector.
# This is equivalent to applying H to |0>.
circuit = QuantumCircuit(1)
circuit.initialize(initial_state, 0)  # Initialize the qubit 0 in initial_state.
circuit.measure_all()

simulate(circuit)
plot_histogram(simulate(circuit))


Now please try to prepare a maximally entangled state $\frac{|00\rangle + |11\rangle}{\sqrt{2}}$ on a two-qubit circuit, and measure the output in the computational basis. Plot the histogram of the measurement outcome.

In [ ]:
## Write your codes here
# ...
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.measure_all()

# simulate(circuit)
plot_histogram(simulate(circuit))

---

### 3. Quantum gates
-   In general, you can easily append a quantum gate to a quantum circuit. The template is `circuit.g(q0)` for single-qubit gates, and `circuit.g(q0, q1)` for two-qubit gates. 
-   Here, `g` is the name of the gate you want to append, usually in lowercase, e.g., `circuit.z(0)` and `circuit.cy(0, 1)` for applying $Z$ gate to qubit 0 and controlled-$Y$ to qubit 0 and 1 respectively.
-   Note that while it is easy to append gates to the end of the circuit, you may find it difficult to insert a gate into the middle of the circuit. So it is advisable to append your gates in a series when constructing your circuit.

An illustrative example is given below. You can try to modify the circuit and see how the output changes.

In [ ]:
circuit = QuantumCircuit(3, 3)
# Here, QuantumCircuit(qb, cb) creates a quantum circuit with qb quantum bits and cb classical bits. Classical bits are usually used as storage for measurement results.
# ----------------------
# Append some gates to your circuit.
circuit.x(0)  # Apply X gate to the first qubit.
circuit.y(1)  # Apply Y gate to the second qubit.
circuit.z(2)  # Apply Z gate to the third qubit.
circuit.h(0)  # Apply H gate to the first qubit.
# circuit.i(0)  # Apply Identity gate to the first qubit.
circuit.s(2)  # Apply S gate to the third qubit.
circuit.sdg(2)  # Apply inverse of S gate to the third qubit.
circuit.t(1)  # Apply T gate to the second qubit.
circuit.rx(np.pi/2, 0)  # Apply RX gate of angle pi/2 to the first qubit.

circuit.cx(0, 1)  # Apply CNOT gate to the first and second qubit.
circuit.swap(0, 1)  # Apply SWAP operation to the first and second qubit.

circuit.ccx(0, 1, 2)  # Apply CCX gate (Toffoli gate) to all three qubits. Qubit 0 and 1 are control bits, qubit 2 is target bit.


circuit.measure(0, 0)  # Measure only the first qubit, the result is stored in the classical bit 0.
circuit.measure([1, 2], [1, 2])  # Measure the last two qubits to last two classical bits.
circuit.draw()


Now we see how to apply some specific gates to qubits in a quantum circuit. What about a more general case? What if we want to apply a unitary gate which is not in the given list of Qiskit?

There are two solutions. 

**(1)** You can specify the 3 rotation angles using the U-gate for all $U \in \mathsf{U}(2)$:
$$
U(\theta,\phi,\lambda) = \left[\begin{matrix}
\cos(\frac{\theta}{2}) & -e^{i\lambda} \sin(\frac{\theta}{2})\\
e^{i\phi} \sin(\frac{\theta}{2}) & e^{i(\phi + \lambda)} \cos(\frac{\theta}{2})
\end{matrix}\right]
$$
Note that taking over all $\theta, \phi, \lambda$ we do traverse all legitimate single-qubit unitary gates.

In [ ]:
circuit = QuantumCircuit(1)
circuit.u(np.pi/3, -np.pi/4, np.pi/2, 0)
circuit.draw()


Alternatively, you can define a matrix by yourself and wrap it as a gate using the method `unitary`. Note that if you want to apply X to the first qubit and Y to the second, you should compute the matrix $Y\otimes X$ instead of the conventional $X\otimes Y$, because Qiskit takes the opposite way.

Let's take CNOT (CX) as an example. You want to apply CNOT to two qubits, setting the first bit as the control bit and the second as the target bit. Recall that in this case,
$$\mathrm{CNOT} = |0\rangle \langle 0| \otimes I + |1\rangle \langle 1| \otimes X$$

In [ ]:
# Define the |0> and |1> basis states.
qb0 = np.array([1, 0])
qb1 = np.array([0, 1])
# Define X
x = np.array([[0, 1],
              [1, 0]])

# Set up the CNOT gate
cnot = np.kron(np.outer(qb0, qb0), np.eye(2)) + np.kron(np.outer(qb1, qb1), x)

u_matrix = Operator(cnot)
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.unitary(u_matrix, [0, 1])  # Apply the CNOT gate to qubit 0 and 1.
circuit.measure_all()
circuit.draw()

---

### 5. Visualizing qubit states using Bloch Sphere
You can visualize a quantum state on the Bloch Sphere. You can specify the two angles $\theta$ and $\phi$ for a single-qubit quantum state $|\psi \rangle$:
$$|\psi \rangle = \cos \frac{\theta}{2} |0\rangle + e^{i\phi}\sin \frac{\theta}{2} |1\rangle$$
Alternatively, you can specify the Cartesian coordinates $x$, $y$, $z$ of the state. For multi-qubit visualization, you can use `plot_bloch_multivector`. This can print a Bloch Sphere for each qubit.

In [ ]:
coords = [1, np.pi/2, 0] # The parameters are passed as [Radius, Theta, Phi]
plot_bloch_vector(coords, coord_type='spherical') # Bloch Vector with spherical coordinates
coords = [1, 0, 0] # Input parameters as [x, y, z] instead
plot_bloch_vector(coords, coord_type='cartesian')

For multi-qubit visualization, you can use `plot_bloch_multivector`. This can print a Bloch Sphere for each qubit.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.x(1)
state = Statevector(qc)
plot_bloch_multivector(state)

---

### 6. Exercises

**Note:** For the following exercises, the initial qubits of the circuits must be in the ground state $|0\rangle$ and you cannot use `circuit.initialize` to initialize a qubit. In real experiments, we almost always start with ground states and design circuits to prepare the state we want.

* Implement 4 circuits to create 4 Bell states $|B_i\rangle, i=0,1,2,3$. Print out the final state vector.
    (Hint: you can use `circuit.save_statevector()` to save the state vector. Then you can use `result.get_statevector()` to obtain it.)

In [ ]:
## Write your codes here
# ...
operators = ['id', 'x', 'y', 'z']
for op in operators:
    circuit = QuantumCircuit(2)
    circuit.h(0)
    circuit.cx(0, 1)
    eval(f'circuit.{op}(0)')
    print(circuit)
    print(np.array(Statevector(circuit)))

* Create a circuit to prepare the state $|\psi\rangle = |+\rangle$ and calculate the expectation of $H = (|0\rangle \langle 0| - |1\rangle \langle 1|) / 2$ with respect to the pure state $|\psi\rangle$.

In [ ]:
## Write your codes here
# ...
basis0 = np.array([1, 0])
basis1 = np.array([0, 1])
H = (np.outer(basis0, basis0) - np.outer(basis1, basis1)) / 2


circuit = QuantumCircuit(1, 1)
circuit.h(0)
print(Statevector(circuit))
# The eigenvalues and eigenvectors of observable H
eigvals, u = np.linalg.eigh(H)
udg = u.conj().T
# Add the additional gate to circuit
circuit.unitary(udg, 0)
circuit.measure(0, 0)
# Run simulation
shots = 10000
backend = Aer.get_backend('aer_simulator')
result = backend.run(circuit, shots=shots).result()  # We run the circuit 1000 times
counts = result.get_counts()
expectation = sum(map(lambda item: eigvals[int(item[0], 2)] * item[1] / shots, counts.items()))
expectation

* Design a circuit that generates the state:
    $$
    |\psi\rangle = \frac{1}{\sqrt{2^{6}}}\sum_{x = 0}^{2^{6} - 1} |x\rangle
    $$
    Measure all qubits in the computational basis. Do you get all your results with equal probability?

In [ ]:
## Write your codes here
# ...
circuit = QuantumCircuit(6)
for i in range(5):
    circuit.h(i)
circuit.measure_all()
circuit.draw()
plot_histogram(simulate(circuit))

* Design a circuit for the transformation:
    $$
    |1\rangle \to \frac{1}{2} (|00\rangle + e^{i\pi / 2}|01 \rangle - |10\rangle + e^{i 3\pi / 2} |11\rangle)
    $$
    hint: you may first decompose the state for actions to reach this expression.

#### Learn to "undo" the quantum state
Note that we can rewrite the final state (up to normalization) as
$$
\begin{aligned}
|00\rangle + e^{i\pi / 2}|01 \rangle - |10\rangle + e^{i 3\pi / 2} |11\rangle &= \ket{00} + i \ket{01} - \ket{10} - i \ket{11} \\
&= \ket{00} - \ket{10} + i\left(\ket{01} - \ket{11} \right) \\
&= CY(2, 1)\left[\ket{00} - \ket{10} - \ket{11} - \ket{01} \right] \\
&= CY(2, 1)\left[\ket{0}(\ket{0} - \ket{1}) - \ket{1}(\ket{0} + \ket{1}) \right] \\
&\propto CY(2, 1)\left[\ket{0}(\ket{0} - \ket{1}) - \ket{1} Z(\ket{0} - \ket{1}) \right] \\
&= CY(2, 1) CZ(1, 2) \left[(\ket{0} - \ket{1}) (\ket{0} - \ket{1}) \right] \\
&\propto CY(2, 1) CZ(1, 2) H^{\otimes 2}(\ket{11})
\end{aligned}
$$

In [ ]:
## Write your codes here
# ...
# Prepare the |11> initial state
circuit = QuantumCircuit(2)
circuit.x(0)
circuit.x(1)

# Apply the transformation
circuit.h(0)
circuit.h(1)
circuit.cz(0, 1)
circuit.cy(1, 0)
Statevector(circuit)